# Model Validation

This notebook performs external validation of a best pre-trained transformer model from notebook 1-TransformerComparison on an unknown validation dataset.
It evaluates the model's performance using metrics such as accuracy, recall, ROC-AUC, and confusion matrix.

In [ ]:
import torch
import pandas as pd
import numpy as np
import datetime
import os
import logging
import random
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, recall_score
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, set_seed

### Configuration and Seed Setting

In [ ]:
SEED = 42

def set_all_seeds(seed: int = SEED, deterministic: bool = False):
    """
    Set all random seeds for reproducibility across Python, NumPy, PyTorch, and Transformers.

    Args:
        seed: Random seed value to use for all random number generators.
        deterministic: If True, enables deterministic CUDA operations. This may reduce performance
                      but ensures full reproducibility on GPU. Note that some operations may not
                      have deterministic implementations.

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    set_seed(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(SEED, deterministic=False)

### Error Logging Setup

In [ ]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

log_file = os.path.join(log_dir, "error_log.csv")

logging.basicConfig(level=logging.ERROR,
                    format="%(asctime)s,%(levelname)s,%(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")

def log_error(error_message, model_name="Unknown"):
    """
    Log error messages to a CSV file with timestamp and model information.

    Args:
        error_message: The error message or exception details to log.
        model_name: Name or identifier of the model where the error occurred.

    Returns:
        None
    """

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if not os.path.exists(log_file):
        df = pd.DataFrame(columns=["timestamp", "model_name", "error_message"])
        df.to_csv(log_file, index=False)

    df = pd.DataFrame([[timestamp, model_name, error_message]],
                      columns=["timestamp", "model_name", "error_message"])
    df.to_csv(log_file, mode="a", header=False, index=False)

### Paths and Relatives

In [ ]:
MODEL_PATH = "../models/FacebookAI_roberta-base"
OUTPUT_DIR = "../models/ExternalValidation_RobertaBase"

DATA_PATH = "../data/input/ExternalValidationDataset.csv"
DATA_ENCODING = "latin1"
DATA_SEP = ";"

LABEL_COL = "GDPR-criticality"
TEXT_COL = "Concatenation"


### Read Eval Data Set

In [ ]:
raw_data = pd.read_csv(DATA_PATH, encoding=DATA_ENCODING, sep=DATA_SEP).dropna(axis=1)
input_data = raw_data.copy(deep=True)

input_data = input_data.rename(columns={TEXT_COL: "text", LABEL_COL: "labels"})
input_data["text"] = input_data["text"].fillna("").astype(str)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


datasets = DatasetDict({
    "train": Dataset.from_pandas(pd.concat([input_data['text'], input_data['labels']], axis=1)),
    "test": Dataset.from_pandas(pd.concat([input_data['text'], input_data['labels']], axis=1)),
    "val": Dataset.from_pandas(pd.concat([input_data['text'], input_data['labels']], axis=1))
})

### Model Definition and Evaluation


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def preprocess_function(examples):
    """
    Tokenize text examples for transformer model input.

    Args:
        examples: Dictionary containing 'text' field with input texts.

    Returns:
        Dictionary with tokenized inputs (input_ids, attention_mask, etc.).
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_data = datasets.map(preprocess_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=2,
    torch_dtype="auto",
    device_map="cuda" if torch.cuda.is_available() else None,
    ignore_mismatched_sizes=True
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "recall": recall_score(labels, preds, average="binary", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    seed=SEED,
    data_seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["val"],
    compute_metrics=compute_metrics
)


### Use Model to predict on external Data

In [ ]:
test_predictions = trainer.predict(tokenized_data["test"])
logits = test_predictions.predictions

probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
p_crit = probs[:, 1]

y_pred = np.argmax(logits, axis=1)
y_true = np.array(tokenized_data["test"]["labels"])

### Display Classification Metrics

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=[0,1])
disp.plot()
plt.show()

print('\n\nClassification Report:\n')
print(classification_report(y_true, y_pred))

print('\nAUC: ', roc_auc_score(y_true, y_pred))
print('\nRecall Score: ', recall_score(y_true, y_pred, average="macro"))